Inspired by Bereta & Diamantis (2025), "The Shape of Consumer Behavior:
A Symbolic and Topological Analysis of Time Series"
 
Pipeline stages
----------------
1.  Data loading        - reads gt_stitched_fixed_*.csv for every term folder
2.  KS test              - Kolmogorov-Smirnov test for normality (paper Table 3)
3.  Output               - CSVs + plots replicating the paper's tables/figures

In [37]:
from __future__ import annotations
 
import warnings
from pathlib import Path
from dataclasses import dataclass, field
 
import numpy as np
import pandas as pd
import matplotlib
 
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
warnings.filterwarnings("ignore")
 
# ----------------------------------------------------------------------------
# CONFIG -- edit these for your environment
# ----------------------------------------------------------------------------
 
DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\plots")
 
# Stitched files live at: DATA_DIR/<term>/stitched/gt_fixed_<term>_<start>_<end>.csv
# (NOT directly inside the term folder, and the filename embeds the term name
# itself plus a date range that varies, so it can't be matched by a fixed
# literal filename -- it's located via the "stitched" subfolder + glob below.)
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed_*.csv"
 
# --- STL decomposition / ADF test (paper Section 3.2-3.4) ---
ROLLING_WINDOW = 91          # 13-week rolling avg (paper) -> 13*7 = 91 days
STL_PERIOD = 365              # seasonal period: paper used weekly seasonality
                               # implicitly tied to ~annual cycle; daily data ->
                               # 365-day period captures the same annual seasonality
ADF_ALPHA = 0.05               # significance level for ADF stationarity test

In [38]:
# ----------------------------------------------------------------------------
# 1. DATA LOADING
# ----------------------------------------------------------------------------
 
def load_all_series(
    data_dir: Path,
    stitched_subdir: str = STITCHED_SUBDIR,
    filename_glob: str = STITCHED_GLOB,
) -> dict[str, pd.Series]:
    """
    Walk every term subfolder in data_dir, read its fixed stitched daily csv,
    and return {term_name: pandas Series indexed by date}.
    """

    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []

    for folder in sorted(data_dir.iterdir()):

        if not folder.is_dir():
            continue

        stitched_dir = folder / stitched_subdir

        if not stitched_dir.exists():
            failed.append((folder.name, f"missing subfolder {stitched_subdir}"))
            continue

        matches = sorted(stitched_dir.glob(filename_glob))

        if not matches:
            failed.append((folder.name, f"missing file matching {filename_glob}"))
            continue

        if len(matches) > 1:
            matches = sorted(
                matches,
                key=lambda p: p.stat().st_mtime,
                reverse=True
            )

        fpath = matches[0]

        try:
            df = pd.read_csv(
                fpath,
                parse_dates=["Time"]
            )

            value_cols = [
                c for c in df.columns
                if c != "Time"
            ]

            if len(value_cols) != 1:
                raise ValueError(
                    f"expected 1 value column, found {len(value_cols)}: {value_cols}"
                )

            value_col = value_cols[0]

            ts = (
                df
                .set_index("Time")[value_col]
                .sort_index()
                .astype(float)
            )

            ts.name = folder.name
            series_dict[folder.name] = ts

        except Exception as e:
            failed.append((folder.name, str(e)))

    if failed:
        print(f"[load_all_series] {len(failed)} terms failed to load:")
        for name, err in failed:
            print(f"    {name}: {err}")

    print(f"[load_all_series] Loaded {len(series_dict)} term series.")

    return series_dict
 
 
def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all series on their shared date index (outer join), as in paper Fig.5/Table 1."""
    panel = pd.DataFrame(series_dict)
    return panel
 
 
def summary_statistics(panel: pd.DataFrame) -> pd.DataFrame:
    """Replicates paper Table 1 (count, mean, std, min, 25/50/75pct, max) for every term."""
    desc = panel.describe(percentiles=[0.25, 0.5, 0.75]).T
    desc = desc.rename(columns={
        "count": "Count", "mean": "Mean", "std": "Std", "min": "Min",
        "25%": "25%", "50%": "50%", "75%": "75%", "max": "Max",
    })
    return desc[["Count", "Mean", "Std", "Min", "25%", "50%", "75%", "Max"]]

In [39]:
# ----------------------------------------------------------------------------
# 2. KOLMOGOROV-SMIRNOV TEST FOR NORMALITY (paper Table 3 / Section 3)
# ----------------------------------------------------------------------------
 
from scipy import stats


def ks_normality_test(panel: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    For each term's full raw series, run a one-sample KS test against a
    normal distribution fitted to that series' own mean/std (matching the
    paper's approach in Section 3 / Table 3).
    """
    rows = []
    for term in panel.columns:
        x = panel[term].dropna().values
        mu, sigma = x.mean(), x.std(ddof=1)
        if sigma == 0:
            stat, p = np.nan, np.nan
        else:
            stat, p = stats.kstest(x, "norm", args=(mu, sigma))
        rows.append({
            "Keyword": term,
            "KS Statistic": stat,
            "p-value": p,
            "Reject H0 at 5%": "Yes" if (p is not np.nan and p < alpha) else "No",
        })
    out = pd.DataFrame(rows).sort_values("KS Statistic", ascending=False).reset_index(drop=True)
    return out

def plot_ks_bar(ks_df: pd.DataFrame, outpath: Path):
    df = ks_df.sort_values("KS Statistic")
    colors = ["tab:red" if r == "Yes" else "tab:green" for r in df["Reject H0 at 5%"]]
    plt.figure(figsize=(10, max(6, 0.18 * len(df))))
    plt.barh(df["Keyword"], df["KS Statistic"], color=colors)
    plt.xlabel("KS Statistic")
    plt.title("KS Test for Normality per Keyword (red = reject H0 at 5%)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

In [40]:
# ----------------------------------------------------------------------------
# 2b. DESCRIPTIVE STATISTICS  (paper Section 3.3: boxplots, volatility, top interest)
# ----------------------------------------------------------------------------
 
def descriptive_statistics_extras(panel: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Replicates paper Section 3.3 / Figure 9 / Figure 10:
      - per-keyword distribution (boxplot data already in summary_statistics)
      - top-5 most volatile keywords (by std)
      - top-5 keywords by highest average interest (by mean)
    """
    stds = panel.std().sort_values(ascending=False)
    means = panel.mean().sort_values(ascending=False)
 
    top5_volatile = stds.head(5).rename("Std").to_frame()
    top5_interest = means.head(5).rename("Mean").to_frame()
 
    return {
        "stds_all": stds.rename("Std").to_frame(),
        "means_all": means.rename("Mean").to_frame(),
        "top5_volatile": top5_volatile,
        "top5_interest": top5_interest,
    }
 
 
def plot_boxplots(panel: pd.DataFrame, outpath: Path, terms_per_row: int = 30):
    """
    Replicates paper Figure 9. With more than 100 terms, a single boxplot row would be
    unreadable, so terms are split into chunks of `terms_per_row` and plotted
    across multiple stacked subplots instead of one wide figure.
    """
    terms = list(panel.columns)
    n_chunks = int(np.ceil(len(terms) / terms_per_row))
    fig, axes = plt.subplots(n_chunks, 1, figsize=(16, 4 * n_chunks))
    if n_chunks == 1:
        axes = [axes]
    for i in range(n_chunks):
        chunk = terms[i * terms_per_row:(i + 1) * terms_per_row]
        ax = axes[i]
        ax.boxplot([panel[t].dropna().values for t in chunk], label=chunk, showfliers=True)
        ax.set_xticklabels(chunk, rotation=90, fontsize=7)
        ax.set_ylabel("Interest (0-100)")
    fig.suptitle("Boxplot of Interest per Keyword")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()
 
 
def plot_top5_volatility_interest(stats_extras: dict[str, pd.DataFrame], outpath: Path):
    """Replicates paper Figure 10: top-5 volatile keywords + top-5 highest-interest keywords."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
 
    vol = stats_extras["top5_volatile"].iloc[::-1]
    axes[0].barh(vol.index, vol["Std"], color="tab:orange")
    axes[0].set_xlabel("Standard Deviation")
    axes[0].set_title("Most Volatile Keywords")
 
    interest = stats_extras["top5_interest"].iloc[::-1]
    axes[1].barh(interest.index, interest["Mean"], color="tab:blue")
    axes[1].set_xlabel("Mean Interest")
    axes[1].set_title("Highest Average Interest")
 
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

In [41]:
# ----------------------------------------------------------------------------
# 2c. SEASONAL-TREND DECOMPOSITION (STL)  (paper Section 3.4)
# ----------------------------------------------------------------------------
 
def run_stl_decomposition(series: pd.Series, period: int = STL_PERIOD) -> STL:
    """
    Runs STL (Seasonal-Trend decomposition using LOESS) on a single series.
    Returns the fitted DecomposeResult (has .trend, .seasonal, .resid, .observed).
    Requires at least 2 full periods of data; raises if the series is too short.
    """
    x = series.dropna()
    if len(x) < 2 * period:
        raise ValueError(
            f"Series too short for STL with period={period} "
            f"(need >= {2 * period} points, have {len(x)})."
        )
    stl = STL(x, period=period, robust=True)
    return stl.fit()
 
 
def run_stl_for_all(series_dict: dict[str, pd.Series], period: int = STL_PERIOD,
                     outdir: Path | None = None) -> dict[str, object]:
    """
    Runs STL decomposition for every term. Optionally saves a 4-panel plot
    (observed/trend/seasonal/resid) per term, mirroring paper Figure 11.
    Returns dict[term] -> statsmodels DecomposeResult.
    """
    results = {}
    failed = []
    for term, s in series_dict.items():
        try:
            res = run_stl_decomposition(s, period)
            results[term] = res
            if outdir is not None:
                plot_stl_decomposition(term, res, outdir / f"stl_{term}.png")
        except Exception as e:
            failed.append((term, str(e)))
    if failed:
        print(f"[STL] {len(failed)} terms failed:")
        for t, e in failed:
            print(f"    {t}: {e}")
    return results
 
 
def plot_stl_decomposition(term: str, result, outpath: Path):
    """Replicates paper Figure 11: 4-panel STL decomposition plot for one keyword."""
    fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
    axes[0].plot(result.observed, linewidth=0.8)
    axes[0].set_ylabel("Observed")
    axes[0].set_title(f"STL Decomposition: {term}")
 
    axes[1].plot(result.trend, linewidth=0.8, color="tab:orange")
    axes[1].set_ylabel("Trend")
 
    axes[2].plot(result.seasonal, linewidth=0.8, color="tab:green")
    axes[2].set_ylabel("Season")
 
    axes[3].scatter(result.resid.index, result.resid.values, s=4, color="tab:blue")
    axes[3].axhline(0, color="black", linewidth=0.6)
    axes[3].set_ylabel("Resid")
    axes[3].set_xlabel("Date")
 
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()
 
 
def stl_summary_table(stl_results: dict[str, object]) -> pd.DataFrame:
    rows = []

    for term, res in stl_results.items():
        observed_var = np.nanvar(res.observed)
        trend_var = np.nanvar(res.trend)
        season_var = np.nanvar(res.seasonal)
        resid_var = np.nanvar(res.resid)

        rows.append({
            "Keyword": term,
            "Observed Var": observed_var,
            "Trend Var": trend_var,
            "Seasonal Var": season_var,
            "Residual Var": resid_var,
            "Seasonal Strength": (
                max(0, 1 - resid_var / (season_var + resid_var))
                if (season_var + resid_var) > 0
                else np.nan
            ),
            "Trend Strength": (
                max(0, 1 - resid_var / (trend_var + resid_var))
                if (trend_var + resid_var) > 0
                else np.nan
            ),
        })

    cols = [
        "Keyword",
        "Observed Var",
        "Trend Var",
        "Seasonal Var",
        "Residual Var",
        "Seasonal Strength",
        "Trend Strength",
    ]

    if not rows:
        return pd.DataFrame(columns=cols)

    return (
        pd.DataFrame(rows)
        .sort_values("Keyword")
        .reset_index(drop=True)
    )

In [42]:
# ----------------------------------------------------------------------------
# 2d. AUGMENTED DICKEY-FULLER TEST  (paper Section 3.4 / Table 4 / Figure 12)
# ----------------------------------------------------------------------------
 
def adf_test_all(panel: pd.DataFrame, alpha: float = ADF_ALPHA) -> pd.DataFrame:
    """
    Replicates paper Table 4: runs the Augmented Dickey-Fuller test on each
    term's full raw series. H0: series has a unit root (non-stationary).
    Rejects H0 (i.e. series IS stationary) if p-value < alpha.
    """
    rows = []
    for term in panel.columns:
        x = panel[term].dropna().values
        try:
            adf_stat, p_value, *_ = adfuller(x, autolag="AIC")
        except Exception as e:
            adf_stat, p_value = np.nan, np.nan
        rows.append({
            "Keyword": term,
            "ADF Statistic": adf_stat,
            "p-value": p_value,
            "Stationary": "Yes" if (p_value is not np.nan and p_value < alpha) else "No",
        })
    out = pd.DataFrame(rows).sort_values("ADF Statistic").reset_index(drop=True)
    return out
 
 
def plot_adf_results(adf_df: pd.DataFrame, outpath: Path):
    """Replicates paper Figure 12: bar chart of ADF statistics, green=stationary, red=not."""
    df = adf_df.sort_values("ADF Statistic")
    colors = ["tab:green" if s == "Yes" else "tab:red" for s in df["Stationary"]]
    plt.figure(figsize=(10, max(6, 0.16 * len(df))))
    plt.bar(df["Keyword"], df["ADF Statistic"], color=colors)
    plt.xticks(rotation=90, fontsize=6)
    plt.ylabel("ADF Statistic")
    plt.title("ADF Test Results by Keyword (green = stationary at 5%)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

In [43]:
# ----------------------------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------------------------
 
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
    # ---- 1. Load data ----
    print("=" * 70)
    print("STEP 1: Loading data")
    print("=" * 70)
    series_dict = load_all_series(DATA_DIR, STITCHED_SUBDIR, STITCHED_GLOB)
    panel = build_panel(series_dict)
    summary_statistics(panel).to_csv(OUTPUT_DIR / "table1_summary_statistics.csv")
 
    # ---- 2. KS test ----
    print("\n" + "=" * 70)
    print("STEP 2: Kolmogorov-Smirnov normality test")
    print("=" * 70)
    ks_df = ks_normality_test(panel)
    ks_df.to_csv(OUTPUT_DIR / "table3_ks_test_results.csv", index=False)
    plot_ks_bar(ks_df, OUTPUT_DIR / "fig_ks_test.png")
    print(ks_df.head(10).to_string(index=False))
 
    # ---- 2b. Descriptive statistics extras ----
    print("\n" + "=" * 70)
    print("STEP 2b: Descriptive statistics (boxplots, volatility, top interest)")
    print("=" * 70)
    desc_extras = descriptive_statistics_extras(panel)
    desc_extras["stds_all"].to_csv(OUTPUT_DIR / "descriptive_stds_all.csv")
    desc_extras["means_all"].to_csv(OUTPUT_DIR / "descriptive_means_all.csv")
    desc_extras["top5_volatile"].to_csv(OUTPUT_DIR / "fig10_top5_volatile.csv")
    desc_extras["top5_interest"].to_csv(OUTPUT_DIR / "fig10_top5_interest.csv")
    plot_boxplots(panel, OUTPUT_DIR / "fig9_boxplots_per_keyword.png")
    plot_top5_volatility_interest(desc_extras, OUTPUT_DIR / "fig10_top5_volatility_interest.png")
    print("Top 5 most volatile keywords:")
    print(desc_extras["top5_volatile"].to_string())
    print("Top 5 keywords by mean interest:")
    print(desc_extras["top5_interest"].to_string())
 
    # ---- 2c. STL decomposition ----
    print("\n" + "=" * 70)
    print("STEP 2c: STL seasonal-trend decomposition")
    print("=" * 70)
    stl_dir = OUTPUT_DIR / "stl_plots"
    stl_dir.mkdir(parents=True, exist_ok=True)
    stl_results = run_stl_for_all(series_dict, STL_PERIOD, outdir=stl_dir)
    stl_table = stl_summary_table(stl_results)
    stl_table.to_csv(OUTPUT_DIR / "stl_summary_table.csv", index=False)
    print(f"STL decomposition completed for {len(stl_results)}/{len(series_dict)} terms.")
    print(f"Per-term 4-panel plots saved to: {stl_dir}")
 
    # ---- 2d. ADF test ----
    print("\n" + "=" * 70)
    print("STEP 2d: Augmented Dickey-Fuller stationarity test")
    print("=" * 70)
    adf_df = adf_test_all(panel)
    adf_df.to_csv(OUTPUT_DIR / "table4_adf_test_results.csv", index=False)
    plot_adf_results(adf_df, OUTPUT_DIR / "fig12_adf_test_results.png")
    n_stationary = (adf_df["Stationary"] == "Yes").sum()
    print(f"{n_stationary}/{len(adf_df)} keywords are stationary at the 5% level.")
    print(adf_df.head(10).to_string(index=False))
 
    print(f"\nAll outputs written to: {OUTPUT_DIR}")
 
 
if __name__ == "__main__":
    main()

STEP 1: Loading data
[load_all_series] Loaded 191 term series.

STEP 2: Kolmogorov-Smirnov normality test
                                 Keyword  KS Statistic       p-value Reject H0 at 5%
                     men‘s_march_madness      0.523194  0.000000e+00             Yes
                      groundhog_day_2023      0.513171  0.000000e+00             Yes
2024_united_states_presidential_election      0.509291  0.000000e+00             Yes
                arizona_election_results      0.492111  0.000000e+00             Yes
                     full_moon_june_2022      0.491860 1.903010e-313             Yes
                    australian_open_2022      0.478355  0.000000e+00             Yes
             when_we_were_young_festival      0.451864 4.925822e-301             Yes
                             fathers_day      0.444376 1.245987e-290             Yes
                         canelo_vs_bivol      0.443480 2.116628e-289             Yes
                        st_patrick's_day    